In [1]:
import sys
sys.path.append("/home/jovyan/work")

from src.utils.spark_session import get_spark_session
from src.utils.lakehouse import read_table, write_table
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType
from datetime import datetime, timedelta
import random

spark = get_spark_session("whatsapp-receipt-agents")
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

random.seed(7)

### Bronze Layer
#### Whatsapp agents capture raw data

In [2]:
# raw messages (simulated)

vehicles_sample = read_table(spark, "silver", "vehicle").select(
    "vehicle_id", "auction_lot_no", "make", "model"
).orderBy("vehicle_id").limit(14).collect()

drivers = read_table(spark, "silver", "staff").filter("staff_id in (2,3)").select("staff_id", "name", "phone_whatsapp").collect()
mechanics = read_table(spark, "silver", "staff").filter("staff_id in (4,5)").select("staff_id", "name", "phone_whatsapp").collect()

now = datetime.now()
messages = []

# clear, lot-number-referenced messages -- should extract with high confidence
for v in vehicles_sample[0:4]:
    d = random.choice(drivers)
    messages.append((d.phone_whatsapp, f"Just picked up the {v.make} {v.model} ({v.auction_lot_no}) from the port, heading to the office now."))

for v in vehicles_sample[4:8]:
    d = random.choice(drivers)
    messages.append((d.phone_whatsapp, f"{v.make} {v.model} ({v.auction_lot_no}) has arrived at the office."))

for v in vehicles_sample[8:12]:
    m = random.choice(mechanics)
    messages.append((m.phone_whatsapp, f"Fixed the bumper on {v.auction_lot_no}, moving on to the headlight tomorrow."))

# ambiguous -- no lot number, only a vague description
messages.append((drivers[0].phone_whatsapp, f"The {vehicles_sample[12].make} {vehicles_sample[12].model} is at the office now."))
messages.append((drivers[1].phone_whatsapp, "That Toyota from yesterday is looking good after the wash."))

# irrelevant chatter -- no vehicle reference at all
messages.append((drivers[0].phone_whatsapp, "Good morning everyone, safe journey today"))
messages.append((mechanics[0].phone_whatsapp, "Happy new month guys!"))

raw_message_schema = StructType([
    StructField("message_id", LongType(), False),
    StructField("group_id", LongType(), False),
    StructField("sender_phone", StringType(), True),
    StructField("sent_at", TimestampType(), False),
    StructField("message_text", StringType(), True),
    StructField("ingested_at", TimestampType(), False),
])

raw_message_rows = [
    Row(
        message_id=i,
        group_id=1,
        sender_phone=phone,
        sent_at=now - timedelta(hours=random.randint(1, 48)),
        message_text=text,
        ingested_at=now,
    )
    for i, (phone, text) in enumerate(messages, start=1)
]

df_raw_message = spark.createDataFrame(raw_message_rows, schema=raw_message_schema)
write_table(df_raw_message, "bronze", "raw_message")

print(f"bronze.raw_message: {read_table(spark, 'bronze', 'raw_message').count()} rows")
read_table(spark, "bronze", "raw_message").orderBy("sent_at").toPandas()

bronze.raw_message: 16 rows


,message_id,group_id,sender_phone,sent_at,message_text,ingested_at
0,11,1,+2348030000005,2026-07-18 20:22:41.411562,"Fixed the bumper on LOT-78713, moving on to th...",2026-07-20 13:22:41.411562
1,12,1,+2348030000004,2026-07-18 20:22:41.411562,"Fixed the bumper on LOT-51834, moving on to th...",2026-07-20 13:22:41.411562
2,13,1,+2348030000002,2026-07-18 23:22:41.411562,The Toyota Corolla is at the office now.,2026-07-20 13:22:41.411562
3,16,1,+2348030000005,2026-07-18 23:22:41.411562,Happy new month guys!,2026-07-20 13:22:41.411562
4,8,1,+2348030000002,2026-07-19 00:22:41.411562,Nissan Murano (LOT-14627) has arrived at the o...,2026-07-20 13:22:41.411562
5,15,1,+2348030000002,2026-07-19 00:22:41.411562,"Good morning everyone, safe journey today",2026-07-20 13:22:41.411562
6,5,1,+2348030000002,2026-07-19 01:22:41.411562,Nissan Altima (LOT-83503) has arrived at the o...,2026-07-20 13:22:41.411562
7,6,1,+2348030000002,2026-07-19 09:22:41.411562,Lexus RX350 (LOT-42388) has arrived at the off...,2026-07-20 13:22:41.411562
8,1,1,+2348030000003,2026-07-19 10:22:41.411562,Just picked up the Ford Explorer (LOT-01338) f...,2026-07-20 13:22:41.411562
9,3,1,+2348030000003,2026-07-19 21:22:41.411562,Just picked up the Ford Edge (LOT-61849) from ...,2026-07-20 13:22:41.411562


In [3]:
# Extracted Events

from pyspark.sql.types import DoubleType

open_vehicles = read_table(spark, "silver", "vehicle").select("vehicle_id", "auction_lot_no", "make", "model").collect()

def classify_event_type(text):
    t = text.lower()
    if "picked up" in t or "picking up" in t:
        return "PICKUP"
    if "arrived" in t or "at the office" in t:
        return "OFFICE_ARRIVAL"
    if "fixed" in t or "bumper" in t or "headlight" in t or "repair" in t or "working on" in t:
        return "REPAIR_PROGRESS"
    if "held" in t or "duty" in t or "customs" in t:
        return "CUSTOMS_UPDATE"
    return "IGNORE"

def extract(text):
    event_type = classify_event_type(text)
    if event_type == "IGNORE":
        return event_type, None, 0.10

    for v in open_vehicles:
        if v.auction_lot_no and v.auction_lot_no in text:
            return event_type, v.vehicle_id, 0.95

    candidates = [v for v in open_vehicles if v.make.lower() in text.lower() and v.model.lower() in text.lower()]
    if len(candidates) == 1:
        return event_type, candidates[0].vehicle_id, 0.60
    elif len(candidates) > 1:
        return event_type, None, 0.35
    else:
        return event_type, None, 0.30

raw_messages = read_table(spark, "bronze", "raw_message").collect()

extracted_rows = []
for m in raw_messages:
    event_type, vehicle_id, confidence = extract(m.message_text)
    review_status = "AUTO_ACCEPTED" if confidence >= 0.85 else ("AUTO_REJECTED" if event_type == "IGNORE" else "PENDING")
    extracted_rows.append(Row(
        event_id=m.message_id, message_id=m.message_id, vehicle_id=vehicle_id,
        event_type=event_type,
        extracted_fields=f'{{"raw_text": "{m.message_text}"}}',
        confidence_score=confidence, review_status=review_status,
        reviewed_by_staff_id=None, reviewed_at=None,
    ))

extracted_event_schema = StructType([
    StructField("event_id", LongType(), False),
    StructField("message_id", LongType(), False),
    StructField("vehicle_id", LongType(), True),
    StructField("event_type", StringType(), True),
    StructField("extracted_fields", StringType(), True),
    StructField("confidence_score", DoubleType(), False),
    StructField("review_status", StringType(), False),
    StructField("reviewed_by_staff_id", LongType(), True),
    StructField("reviewed_at", TimestampType(), True),
])

df_extracted = spark.createDataFrame(extracted_rows, schema=extracted_event_schema)
write_table(df_extracted, "silver", "extracted_event")

print(f"silver.extracted_event: {read_table(spark, 'silver', 'extracted_event').count()} rows")
read_table(spark, "silver", "extracted_event").select(
    "event_id", "event_type", "vehicle_id", "confidence_score", "review_status"
).orderBy("event_id").toPandas()

silver.extracted_event: 16 rows


,event_id,event_type,vehicle_id,confidence_score,review_status
0,1,PICKUP,1.0,0.95,AUTO_ACCEPTED
1,2,PICKUP,2.0,0.95,AUTO_ACCEPTED
2,3,PICKUP,3.0,0.95,AUTO_ACCEPTED
3,4,PICKUP,4.0,0.95,AUTO_ACCEPTED
4,5,OFFICE_ARRIVAL,5.0,0.95,AUTO_ACCEPTED
5,6,OFFICE_ARRIVAL,6.0,0.95,AUTO_ACCEPTED
6,7,OFFICE_ARRIVAL,7.0,0.95,AUTO_ACCEPTED
7,8,OFFICE_ARRIVAL,8.0,0.95,AUTO_ACCEPTED
8,9,REPAIR_PROGRESS,9.0,0.95,AUTO_ACCEPTED
9,10,REPAIR_PROGRESS,10.0,0.95,AUTO_ACCEPTED


In [6]:
# review queue (group-message agent)

review_task_schema = StructType([
    StructField("task_id", LongType(), False),
    StructField("source_type", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("assigned_to_staff_id", LongType(), True),
    StructField("status", StringType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("resolved_at", TimestampType(), True),
])

pending_events = read_table(spark, "silver", "extracted_event").filter("review_status = 'PENDING'").select("event_id").collect()

review_task_rows = [
    Row(task_id=i, source_type="EXTRACTED_EVENT", source_id=row.event_id,
        assigned_to_staff_id=1, status="OPEN", created_at=datetime.now(), resolved_at=None)
    for i, row in enumerate(pending_events, start=1)
]

df_review_task = spark.createDataFrame(review_task_rows, schema=review_task_schema)
write_table(df_review_task, "silver", "review_task")

print(f"silver.review_task: {read_table(spark, 'silver', 'review_task').count()} rows")
read_table(spark, "silver", "review_task").toPandas()

silver.review_task: 1 rows


,task_id,source_type,source_id,assigned_to_staff_id,status,created_at,resolved_at
0,1,EXTRACTED_EVENT,13,1,OPEN,2026-07-20 13:35:40.938474,NaT


In [7]:
# Receipt intake agents

from decimal import Decimal
from pyspark.sql.types import DecimalType, DateType

purchases_for_receipts = read_table(spark, "silver", "purchase").select(
    "vehicle_id", "price_amount", "price_currency", "purchase_date"
).orderBy("vehicle_id").limit(10).collect()

owner_phone = read_table(spark, "silver", "staff").filter("staff_id = 1").select("phone_whatsapp").collect()[0].phone_whatsapp

receipt_document_schema = StructType([
    StructField("receipt_id", LongType(), False),
    StructField("sender_phone", StringType(), True),
    StructField("sent_at", TimestampType(), False),
    StructField("file_url", StringType(), False),
    StructField("ingested_at", TimestampType(), False),
])

receipt_rows = []
for i, p in enumerate(purchases_for_receipts, start=1):
    sent_at = datetime.combine(p.purchase_date, datetime.min.time()) + timedelta(hours=random.randint(1, 6))
    receipt_rows.append(Row(
        receipt_id=i, sender_phone=owner_phone, sent_at=sent_at,
        file_url=f"https://placeholder.local/receipts/vehicle_{p.vehicle_id}.jpg",
        ingested_at=sent_at,
    ))

df_receipt_document = spark.createDataFrame(receipt_rows, schema=receipt_document_schema)
write_table(df_receipt_document, "bronze", "receipt_document")
print(f"bronze.receipt_document: {read_table(spark, 'bronze', 'receipt_document').count()} rows")

bronze.receipt_document: 10 rows


In [8]:
#  Simulated OCR extraction

extracted_line_schema = StructType([
    StructField("line_id", LongType(), False),
    StructField("receipt_id", LongType(), False),
    StructField("vehicle_id", LongType(), True),
    StructField("vendor_text", StringType(), True),
    StructField("amount", DecimalType(14, 2), True),
    StructField("currency", StringType(), True),
    StructField("receipt_date", DateType(), True),
    StructField("confidence_score", DoubleType(), False),
    StructField("review_status", StringType(), False),
    StructField("reviewed_by_staff_id", LongType(), True),
    StructField("reviewed_at", TimestampType(), True),
])

VENDOR_NAMES = ["Copart Auto Auctions", "IAAI Salvage", "Manheim Export Desk"]

extracted_line_rows = []
for i, p in enumerate(purchases_for_receipts, start=1):
    ocr_error = random.random() < 0.20
    amount = (p.price_amount * Decimal("1.05")) if ocr_error else p.price_amount
    confidence = 0.55 if ocr_error else 0.92
    review_status = "PENDING" if confidence < 0.85 else "AUTO_ACCEPTED"

    extracted_line_rows.append(Row(
        line_id=i, receipt_id=i, vehicle_id=p.vehicle_id,
        vendor_text=random.choice(VENDOR_NAMES),
        amount=amount, currency=p.price_currency, receipt_date=p.purchase_date,
        confidence_score=confidence, review_status=review_status,
        reviewed_by_staff_id=None, reviewed_at=None,
    ))

df_extracted_line = spark.createDataFrame(extracted_line_rows, schema=extracted_line_schema)
write_table(df_extracted_line, "silver", "extracted_receipt_line")

print(f"silver.extracted_receipt_line: {read_table(spark, 'silver', 'extracted_receipt_line').count()} rows")
read_table(spark, "silver", "extracted_receipt_line").select(
    "line_id", "vehicle_id", "amount", "confidence_score", "review_status"
).orderBy("line_id").toPandas()

silver.extracted_receipt_line: 10 rows


,line_id,vehicle_id,amount,confidence_score,review_status
0,1,1,1771.96,0.55,PENDING
1,2,2,7023.53,0.92,AUTO_ACCEPTED
2,3,3,1825.29,0.55,PENDING
3,4,4,6015.14,0.92,AUTO_ACCEPTED
4,5,5,6759.94,0.92,AUTO_ACCEPTED
5,6,6,3586.43,0.92,AUTO_ACCEPTED
6,7,7,4254.47,0.55,PENDING
7,8,8,2266.58,0.92,AUTO_ACCEPTED
8,9,9,6027.95,0.92,AUTO_ACCEPTED
9,10,10,2436.20,0.92,AUTO_ACCEPTED


In [9]:
# lets review the queue for the low-confidence receipts

pending_receipts = read_table(spark, "silver", "extracted_receipt_line").filter("review_status = 'PENDING'").select("line_id").collect()
existing_max_task_id = read_table(spark, "silver", "review_task").agg(F.max("task_id")).collect()[0][0] or 0

new_review_tasks = [
    Row(task_id=existing_max_task_id + i, source_type="EXTRACTED_RECEIPT_LINE", source_id=row.line_id,
        assigned_to_staff_id=1, status="OPEN", created_at=datetime.now(), resolved_at=None)
    for i, row in enumerate(pending_receipts, start=1)
]

df_new_review_tasks = spark.createDataFrame(new_review_tasks, schema=review_task_schema)
write_table(df_new_review_tasks, "silver", "review_task", mode="append")

print(f"silver.review_task total: {read_table(spark, 'silver', 'review_task').count()} rows")
read_table(spark, "silver", "review_task").groupBy("source_type", "status").count().show()

silver.review_task total: 4 rows
+--------------------+------+-----+
|         source_type|status|count|
+--------------------+------+-----+
|EXTRACTED_RECEIPT...|  OPEN|    3|
|     EXTRACTED_EVENT|  OPEN|    1|
+--------------------+------+-----+



## Fact review_queue
#### The Gold-layer fact that powers the Operations Dashboard - a validation to confirm the message and receipts agents accuracy

In [10]:
from pyspark.sql.types import IntegerType

review_tasks = read_table(spark, "silver", "review_task").collect()
events_conf = {r.event_id: r.confidence_score for r in read_table(spark, "silver", "extracted_event").select("event_id", "confidence_score").collect()}
receipts_conf = {r.line_id: r.confidence_score for r in read_table(spark, "silver", "extracted_receipt_line").select("line_id", "confidence_score").collect()}
staff_key_map = {r.staff_id: r.staff_key for r in read_table(spark, "gold", "dim_staff").filter("is_current = true").select("staff_key", "staff_id").collect()}
source_key_map = {r.source_code: r.source_key for r in read_table(spark, "gold", "dim_data_source").collect()}

fact_review_rows = []
for t in review_tasks:
    conf = events_conf.get(t.source_id) if t.source_type == "EXTRACTED_EVENT" else receipts_conf.get(t.source_id)
    source_code = "WHATSAPP_GROUP_AGENT" if t.source_type == "EXTRACTED_EVENT" else "RECEIPT_AGENT"
    resolution_hours = ((t.resolved_at - t.created_at).total_seconds() / 3600) if t.resolved_at else None

    fact_review_rows.append(Row(
        task_key=t.task_id,
        source_key=source_key_map.get(source_code),
        assigned_staff_key=staff_key_map.get(t.assigned_to_staff_id),
        created_date_key=int(t.created_at.strftime("%Y%m%d")),
        resolved_date_key=int(t.resolved_at.strftime("%Y%m%d")) if t.resolved_at else None,
        confidence_score=conf,
        resolution_hours=resolution_hours,
        status=t.status,
    ))

fact_review_schema = StructType([
    StructField("task_key", LongType(), False),
    StructField("source_key", LongType(), True),
    StructField("assigned_staff_key", LongType(), True),
    StructField("created_date_key", IntegerType(), True),
    StructField("resolved_date_key", IntegerType(), True),
    StructField("confidence_score", DoubleType(), True),
    StructField("resolution_hours", DoubleType(), True),
    StructField("status", StringType(), False),
])

df_fact_review = spark.createDataFrame(fact_review_rows, schema=fact_review_schema)
write_table(df_fact_review, "gold", "fact_review_queue")

print(f"gold.fact_review_queue: {read_table(spark, 'gold', 'fact_review_queue').count()} rows")
read_table(spark, "gold", "fact_review_queue").toPandas()

gold.fact_review_queue: 4 rows


,task_key,source_key,assigned_staff_key,created_date_key,resolved_date_key,confidence_score,resolution_hours,status
0,1,2,1,20260720,NaN,0.35,NaN,OPEN
1,4,3,1,20260720,NaN,0.55,NaN,OPEN
2,3,3,1,20260720,NaN,0.55,NaN,OPEN
3,2,3,1,20260720,NaN,0.55,NaN,OPEN
